# 第三阶段 · Notebook 1：进阶算法综述 — ADAPT-VQE / SQD / SKQD / iQCC

> **阅读（建议先读再跑本笔记本）**：
> - `2512.13657v2.pdf`：化学中的量子优势（iQCC、OLED 分子）
> - `2508.02578v2.pdf`：带可证明收敛性的 SKQD
> - InQuanto 文档：https://docs.quantinuum.com/inquanto/

---

## 为何要在标准 VQE 之外再学这些？

带 **UCCSD** 的标准 **VQE** 有三类主要瓶颈：
1. **参数量过大**：UCCSD 按 $O(N^4)$ 缩放，线路深度增长很快
2. **贫瘠高原（barren plateau）**：深线路下梯度随比特数指数衰减
3. **缺乏收敛保证**：易陷入局部极小

后 VQE 时代的算法针对上述问题设计。时间线（示意）：
```
2016  VQE（Peruzzo 2014）→ 成为主流
2019  ADAPT-VQE：自适应构造 ansatz
2020  QSCI：量子采样驱动的子空间展开
2023  SQD：基于采样的量子对角化（IBM+UCB）
2024  SKQD：Krylov + qDRIFT，可证明收敛
2024  iQCC：工业规模（约 200 逻辑量子比特、OLED 等）
```


---
## 1. ADAPT-VQE：自适应 ansatz 构造

**核心思想**：不一次性固定完整 UCCSD ansatz，而是每次从算符池里选出 **梯度最大** 的激发算符，逐个加入线路。

算法概要：

初始：$|\psi\rangle = |\mathrm{HF}\rangle$
算符池：{全部单/双激发算符 $A_k$}

重复：
  1. 对池中每个 $A_k$ 计算梯度：$g_k = |{\partial\langle H\rangle}/{\partial \theta_k}|_{\theta=0}$
  2. 取 $|g_k|$ 最大的 $A_{k^*}$
  3. 将 $e^{i\theta A_{k^*}}$ 接到 ansatz 上
  4. **重新优化全部**参数
  5. 若 $\max_k |g_k| < \varepsilon$ 则停止


**相对 UCCSD 的优点**：
- 参数量常可减少数倍（只保留“有用”的激发）
- 增长早期不易陷入贫瘠高原
- 可系统逼近 FCI

**代价**：梯度评估次数多（每个算符、每步梯度约需多次线路执行）


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from scipy.optimize import minimize

# H₂ 哈密顿量（Pauli 分解）
H2 = SparsePauliOp.from_list([
    ('II', -1.8505), ('IZ', 0.3980), ('ZI', -0.3980),
    ('ZZ', 0.0112),  ('XX', 0.1807), ('YY', 0.1807),
])
E_FCI = np.linalg.eigvalsh(H2.to_matrix())[0]

# ADAPT-VQE 算符池（2 量子比特，Jordan–Wigner）
# 单激发 a†₁a₀ − h.c. → i(XY−YX)/2；双激发 → iXX, iYY 等
operator_pool = [
    ('XY-YX（单激发）',  SparsePauliOp.from_list([('XY', 0.5), ('YX', -0.5)])),
    ('XX（双激发）',     SparsePauliOp.from_list([('XX', 1.0)])),
    ('YY（双激发）',     SparsePauliOp.from_list([('YY', 1.0)])),
    ('ZZ（对角）',       SparsePauliOp.from_list([('ZZ', 1.0)])),
]

def compute_gradient(state_vec, operator):
    """⟨H⟩ 对算符 A 方向在 θ=0 的梯度相关量：g ∝ Im⟨ψ|[H,A]|ψ⟩"""
    H_mat = H2.to_matrix()
    A_mat = operator.to_matrix()
    psi = state_vec.data
    commutator_ev = psi.conj() @ (H_mat @ A_mat - A_mat @ H_mat) @ psi
    return abs(np.imag(commutator_ev))

def apply_operator_exp(state_vec, operator, theta):
    """对态施加 e^{iθA}（矩阵指数，精确）"""
    A_mat = operator.to_matrix()
    U = np.linalg.matrix_power(
        np.eye(len(A_mat)) * np.cos(theta) + 1j * A_mat * np.sin(theta),
        1
    )
    from scipy.linalg import expm
    U = expm(1j * theta * A_mat)
    new_data = U @ state_vec.data
    return Statevector(new_data)

# ADAPT-VQE 主循环；初态：H₂ 的 HF 参考 |01⟩
psi = Statevector.from_label('01')
selected_ops = []
selected_thetas = []
energy_history = []
gradient_history = []

print('H₂ 上的 ADAPT-VQE：')
print(f'初态（HF）能量: {np.real(psi.data.conj() @ H2.to_matrix() @ psi.data):.6f} Ha')
print(f'FCI 目标:       {E_FCI:.6f} Ha')
print()

H_mat = H2.to_matrix()

for iteration in range(4):
    # 对池中每个算符算梯度
    grads = [(name, compute_gradient(psi, op), op) for name, op in operator_pool]
    grads.sort(key=lambda x: -x[1])

    best_name, best_grad, best_op = grads[0]
    gradient_history.append(best_grad)

    if best_grad < 1e-6:
        print(f'第 {iteration} 轮收敛（最大梯度 = {best_grad:.2e}）')
        break

    print(f'迭代 {iteration+1}：加入算符 [{best_name}]，梯度 = {best_grad:.4f}')
    selected_ops.append(best_op)

    # 重新优化全部参数
    n_params = len(selected_ops)

    def energy_from_params(thetas):
        state = Statevector.from_label('01')
        for op, t in zip(selected_ops, thetas):
            from scipy.linalg import expm
            U = expm(1j * t * op.to_matrix())
            state = Statevector(U @ state.data)
        E = np.real(state.data.conj() @ H_mat @ state.data)
        return E

    x0 = selected_thetas + [0.0]
    result = minimize(energy_from_params, x0, method='COBYLA', options={'maxiter': 500})
    selected_thetas = list(result.x)

    psi = Statevector.from_label('01')
    for op, t in zip(selected_ops, selected_thetas):
        from scipy.linalg import expm
        U = expm(1j * t * op.to_matrix())
        psi = Statevector(U @ psi.data)

    E_current = result.fun
    energy_history.append(E_current)
    error_mHa = abs(E_current - E_FCI) * 1000
    print(f'       能量: {E_current:.6f} Ha  误差: {error_mHa:.3f} mHa  参数数: {n_params}')

print(f'\nADAPT-VQE 最终能量: {energy_history[-1]:.6f} Ha')
print(f'FCI:                {E_FCI:.6f} Ha')
print(f'误差:               {abs(energy_history[-1]-E_FCI)*1000:.3f} mHa')
print(f'UCCSD 约需 4 个参数；本例 ADAPT-VQE 用了 {len(selected_ops)} 个')


---
## 2. SQD — 基于采样的量子对角化（Sample-Based Quantum Diagonalization）

**主要论文**：Robledo-Moreno 等 (2024)，*Nature Chemistry*，arXiv:2405.05068
**IBM 实现**：`qiskit-addon-sqd`

### 核心思想

与 VQE 测期望值不同：**从量子线路采样组态**，构造小而重要的子空间，再在该子空间内 **经典对角化** 哈密顿量。

```
步骤 1：在硬件上运行参数化线路 U(θ)|0⟩，测量比特串 {|b₁⟩,…,|b_K⟩}（K 次采样）

步骤 2：比特串对应电子组态（Slater 行列式），张成子空间 S = span{|b₁⟩,…,|b_K⟩}

步骤 3：将 H 投影到 S（经典）→ H_S = P_S H P_S（仅 K×K 小矩阵）

步骤 4：经典精确对角化 H_S → E₀ 为最低本征值

步骤 5：用所得波函数改进线路参数，迭代至收敛
```

### 相对 VQE 的潜在优势
- 不必做梯度优化（可避开贫瘠高原）
- 线路主要任务是 **找出重要组态**，不必精确编码全部振幅
- 难的对角化交给经典端精确完成
- **在有噪声的硬件上仍可能表现较好**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix

# SQD 核心流程（手写教学版）

def build_fci_hamiltonian(n_qubits, h_one, h_two):
    """
    在占据数基下构造 FCI 哈密顿量（极简示意，二量子化形式简化）。
    """
    dim = 2**n_qubits
    H = np.zeros((dim, dim), dtype=complex)

    for i in range(dim):
        for j in range(dim):
            if i == j:
                occ = [(i >> k) & 1 for k in range(n_qubits)]
                H[i, i] = sum(h_one[k] * occ[k] for k in range(min(n_qubits, len(h_one))))
    return H

def sqd_core(H_full, quantum_samples, n_samples=50):
    """
    SQD：给定线路采样的比特串（此处用组态索引），
    构造子空间哈密顿量并经典对角化。
    """
    n = H_full.shape[0]

    unique_samples = list(set(quantum_samples))[:n_samples]
    K = len(unique_samples)

    H_sub = np.zeros((K, K), dtype=complex)
    for i, si in enumerate(unique_samples):
        for j, sj in enumerate(unique_samples):
            H_sub[i, j] = H_full[si, sj]

    eigenvalues = np.linalg.eigvalsh(H_sub)
    E_sqd = eigenvalues[0]

    return E_sqd, K

# H₂：4×4 全哈密顿矩阵
I = np.eye(2); X = np.array([[0,1],[1,0]]); Y = np.array([[0,-1j],[1j,0]]); Z = np.array([[1,0],[0,-1]])
g = {'II':-1.8505,'IZ':0.3980,'ZI':-0.3980,'ZZ':0.0112,'XX':0.1807,'YY':0.1807}
pm = {'I':I,'X':X,'Y':Y,'Z':Z}
H_full = sum(c*np.kron(pm[s[0]],pm[s[1]]) for s,c in g.items())
E_FCI = np.linalg.eigvalsh(H_full)[0]

# 用不同“采样质量”模拟量子线路：理想 / 均匀随机 / 有偏噪声
np.random.seed(42)
results = {}

_, evecs = np.linalg.eigh(H_full)
gs_probs = np.abs(evecs[:, 0])**2

for circuit_quality, description in [
    (gs_probs, '理想线路（基态精确概率）'),
    (np.ones(4)/4, '均匀随机采样'),
    (np.array([0.05, 0.70, 0.20, 0.05]), '较好但有偏/噪声'),
]:
    energies_vs_K = []
    for K in range(2, 5):
        # 采样约 K 组组态
        samples = np.random.choice(4, size=K*5, replace=True, p=circuit_quality/circuit_quality.sum())
        E_sqd, actual_K = sqd_core(H_full, samples.tolist(), K)
        energies_vs_K.append((K, E_sqd))
    results[description] = energies_vs_K

print('SQD 能量估计 vs 子空间大小 K：')
print(f'{"K":<6}{"理想":<18}{"有偏/噪声":<18}{"均匀随机":<18}')
desc_order = ['理想线路（基态精确概率）', '较好但有偏/噪声', '均匀随机采样']
for i in range(3):
    K = i + 2
    vals = [results[d][i][1] for d in desc_order]
    print(f'{K:<6}' + ''.join(f'{v:<18.4f}' for v in vals))
print(f'FCI 参考: {E_FCI:.6f} Ha')
print()
print('要点：高质量采样时 SQD 往往比 VQE 更快逼近；有噪声时仍能抓住重要组态。')


---
## 3. SKQD：基于采样的 Krylov 量子对角化

**文件夹内论文**：`2508.02578v2.pdf`  
**作者**：Cortes 等 (2025)

> **精读与实现路线**：见**下一格**「3.1 `2508.02578v2.pdf`…」；总览亦见 `learning_materials/README.md` 对应节。

### 与经典 Krylov 方法的关系

经典数值线性代数里常见的 Krylov 思路：
- **Lanczos / Davidson**：FCI 等 CI 求解器常用（MOLPRO、Molcas 等）
- **GMRES、CG**：迭代解线性方程组

由态 $|\psi\rangle$ 与哈密顿量 $H$ 张成的 Krylov 子空间：
$$K_m = \mathrm{span}\{|\psi\rangle, H|\psi\rangle, H^2|\psi\rangle, \ldots, H^{m-1}|\psi\rangle\}$$

**定理（直观）**：将 $H$ 投影到 $K_m$ 上再对角化，随 $m$ 增大可逼近 **与 FCI 一致** 的结果，并有可证明的收敛率。

### SKQD = Krylov + 量子采样

```
步骤 1：用量子线路实现 e^{-iHt}|ψ⟩（qDRIFT 或 Trotter 等），在多个时刻 t₀,…,t_{m−1}

步骤 2：对每个演化态采样（类似 SQD），得到各时刻的比特串快照

步骤 3：由采样组态构造 Krylov 型子空间

步骤 4：在该子空间内经典精确对角化 H

结果：在论文条件下可证明收敛到精确基态能量
```

**qDRIFT**：对哈密顿量项做 **随机化** 的 Trotter 型分解；与 SKQD 结合得到 **SqDRIFT**，并带有理论误差界。


### 3.1 `2508.02578v2.pdf`：精读顺序与实现路径（学习计划增补）

**论文**：*Sample-based Krylov quantum diagonalization*（SKQD）；文中与 **qDRIFT** 结合得到 **SqDRIFT**，针对量子化学哈密顿量给出可证明收敛与数值（如 PAH）。

#### 建议精读顺序（在 PDF 内按节跳转）

1. **Abstract + Introduction** — 目标（NISQ 基态）、与 VQE / 纯 SQD 的差异。
2. **Background** — Krylov 思想；为何用 **$U(t)=e^{-iHt}$** 生成子空间而不是直接用 $H^k$。
3. **SKQD 主算法** — 参考态、$\Delta t$、Krylov 维数 $r$、**采样**、经典投影与对角化。
4. **SqDRIFT** — qDRIFT 如何实现时间演化、误差/深度与 Trotter 对比。
5. **Main theorems** — 收敛条件（对照 QPE 类结果）。
6. **Numerics** — 多环芳烃等尺度与化学相关性。

#### 与本 Notebook 其他格子的关系

| 内容 | 说明 |
|------|------|
| **上一格 §2 SQD** | 采样 → 子空间 → 经典对角化；**无** Krylov 时间演化链。 |
| **下一格 KQD 代码** | 用 **精确** $e^{-iHt_k}$ 在模拟器上构造 Krylov 基并解广义本征问题 — **无量子采样、无 qDRIFT**，用于建立「维数 $m$ ↑ → 能量 → 精确基态」的直觉。 |
| **完整 SKQD/SqDRIFT** | 须按论文把 **采样 + qDRIFT 线路 + 子空间构建** 接起来；见下表。 |

#### 实现路径（由易到难）

| 步骤 | 目标 | 入手处 |
|:---:|:---|:---|
| **A** | Krylov 能量随 $m$ 收敛 | **下一格代码**（H₂ 上的 KQD） |
| **B** | 把 $e^{-iHt}$ 做成 **线路**（Trotter / 乘积公式） | Qiskit：Hamiltonian simulation、`TimeEvolutionProblem` 等 |
| **C** | IBM **SKQD** 教程（与论文同名思路） | IBM Quantum Learning → *Quantum diagonalization algorithms* → **SKQD**；Qiskit 教程 *Sample-based Krylov Quantum Diagonalization*（以当前官网 URL 为准） |
| **D** | **SQD**（相关但非同一件事） | `qiskit-addon-sqd` 与 README / 示例 |
| **E** | **SqDRIFT** 细节 | 论文 `2508.02578v2` + 作者公开代码（若有） |

#### 自检（完成即算「路径走过一遍」）

- [ ] 用荧光笔标出：**定理条件** ↔ 经典 Krylov/CI 里你熟悉的假设。
- [ ] 写 5 行：**SQD vs SKQD** 差在哪一步。
- [ ] （可选）把 **B** 的一条最短 Trotter 线路接到 **A** 的 H₂ 哈密顿量上，比较与 `expm` 的能量差。

---
*中文版总表见本工作区 `learning_materials/README.md` 中节「2508.02578v2（SKQD / SqDRIFT）精读与实现路径」。*

**直链**：arXiv https://arxiv.org/abs/2508.02578 · IBM Quantum Learning https://learning.quantum.ibm.com/（站内搜 SKQD）· Qiskit 文档 https://docs.quantum.ibm.com/（站内搜 *Sample-based Krylov*）


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, eigh

# KQD：模拟器上精确时间演化；对应 2508.02578v2 中的收敛图景（无采样、无 qDRIFT）

# H₂ 哈密顿量
I = np.eye(2); X = np.array([[0,1],[1,0]]); Y = np.array([[0,-1j],[1j,0]]); Z = np.array([[1,0],[0,-1]])
g = {'II':-1.8505,'IZ':0.3980,'ZI':-0.3980,'ZZ':0.0112,'XX':0.1807,'YY':0.1807}
pm = {'I':I,'X':X,'Y':Y,'Z':Z}
H = sum(c*np.kron(pm[s[0]],pm[s[1]]) for s,c in g.items())
E0_exact = np.linalg.eigvalsh(H)[0]

def krylov_qd(H, psi_init, m, t=0.5):
    """
    KQD：用基 {e^{-iHt_k}|ψ⟩} 构造子空间并解广义本征问题（此处为经典精确模拟）。
    """
    n = len(psi_init)

    basis = []
    for k in range(m):
        t_k = k * t
        U_k = expm(-1j * H * t_k)
        phi_k = U_k @ psi_init
        basis.append(phi_k)

    S = np.zeros((m, m), dtype=complex)
    H_kry = np.zeros((m, m), dtype=complex)
    for i in range(m):
        for j in range(m):
            S[i, j]     = basis[i].conj() @ basis[j]
            H_kry[i, j] = basis[i].conj() @ H @ basis[j]

    try:
        eigenvalues = eigh(H_kry, S, eigvals_only=True)
        return eigenvalues[0]
    except np.linalg.LinAlgError:
        return None

# 不同 Krylov 维数 m
psi_HF = np.array([0, 1, 0, 0], dtype=complex)  # |01⟩ = HF
krylov_dims = range(1, 6)
energies_kqd = []

for m in krylov_dims:
    E = krylov_qd(H, psi_HF, m, t=0.3)
    if E is not None:
        energies_kqd.append(E)
    else:
        energies_kqd.append(float('nan'))

# 对比：VQE
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from scipy.optimize import minimize

H2_qiskit = SparsePauliOp.from_list([('II',-1.8505),('IZ',0.3980),('ZI',-0.3980),('ZZ',0.0112),('XX',0.1807),('YY',0.1807)])
theta = Parameter('t')
ansatz = QuantumCircuit(2)
ansatz.x(0); ansatz.ry(2*theta, 0); ansatz.cx(0, 1)
estimator = StatevectorEstimator()
vqe_history = []
def vqe_cost(p):
    E = float(estimator.run([(ansatz, H2_qiskit, [p])]).result()[0].data.evs[0])
    vqe_history.append(E)
    return E
np.random.seed(0)
minimize(vqe_cost, [0.8], method='COBYLA', options={'maxiter':60})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(list(krylov_dims), np.array(energies_kqd), 'b-o', ms=8, lw=2, label='KQD（Krylov 维数 m）')
ax1.axhline(E0_exact, color='r', ls='--', lw=2, label=f'FCI: {E0_exact:.4f} Ha')
ax1.set_xlabel('Krylov 维数 m', fontsize=12)
ax1.set_ylabel('能量 (Hartree)', fontsize=12)
ax1.set_title('KQD 收敛（H₂）', fontsize=13)
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(range(len(vqe_history)), vqe_history, 'g-', lw=1.5, label='VQE（目标函数调用次数）')
ax2.axhline(E0_exact, color='r', ls='--', lw=2, label=f'FCI: {E0_exact:.4f} Ha')
ax2.set_xlabel('VQE 函数求值次数', fontsize=12)
ax2.set_ylabel('能量 (Hartree)', fontsize=12)
ax2.set_title('VQE 收敛（H₂）', fontsize=13)
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('KQD_vs_VQE_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print('KQD 收敛：')
for m, E in zip(krylov_dims, energies_kqd):
    err = abs(E - E0_exact) * 1000 if not np.isnan(E) else float('nan')
    print(f'  m={m}: E={E:.6f} Ha  误差={err:.3f} mHa')
print('SKQD（2508.02578）在此基础上加入：量子采样 + qDRIFT + 可证明误差界')


---
## 4. iQCC：迭代量子比特耦合簇（Iterative Qubit Coupled Cluster）

**文件夹内论文**：`2512.13657v2.pdf`（Towards Quantum Advantage in Chemistry）

### 概览

iQCC 是 **OTI Lumionics / Samsung** 在 OLED 磷光配合物（Ir(III)、Pt(II)）量子化学计算中的核心方法之一。

**特点**：
- 直接在 **量子比特空间** 工作（不必全程用费米子算符形式）
- **哈密顿量 dressing**：用一串 Pauli 旋转逐步相似变换 $H$
- 可在经典端扩展到较大规模（论文量级约 **200 逻辑量子比特** 等效）
- 常配合 **Epstein–Nesbet 微扰修正**（iQCC+PT）

**算法骨架**：
```
1. 从量子比特哈密顿量 H₀ 出发
2. 选最优单 Pauli 旋转 e^{iθP_k}（QCC 步）
3. Dress：H₁ = e^{-iθP_k} H₀ e^{iθP_k}
4. 重复得 H₂, H₃, …, H_n
5. 能量可取 ⟨HF|H_n|HF⟩（最终能量不必再跑量子线路）
6. 对剩余相关做微扰修正
```

### 与你研究方向的联系

2512.13657v2 用 iQCC benchmark：
- Ir(III) 配合物的 **三重激发能 T1**
- 强自旋–轨道耦合与强相关并存
- 与 **多相催化活性位** 上过渡金属体系的电子相关问题 **同类问题**


In [ ]:
import numpy as np

# iQCC：单比特玩具模型，演示 Hamiltonian dressing；完整实现见 InQuanto 文档

# H = a Z + b X
a, b = 0.5, 0.3
Z = np.array([[1, 0], [0, -1]])
X = np.array([[0, 1], [1, 0]])
H = a * Z + b * X

E_exact = np.linalg.eigvalsh(H)[0]
print(f'精确基态能量: {E_exact:.6f}')
print(f'HF（|0⟩）能量: {H[0,0]:.6f}')

Y = np.array([[0, -1j], [1j, 0]], dtype=complex)

thetas = np.linspace(-np.pi, np.pi, 200)
H_dressed_energies = []
for t in thetas:
    U = np.cos(t)*np.eye(2) + 1j*np.sin(t)*Y
    H_dressed = U.conj().T @ H @ U
    E_iqcc = np.real(H_dressed[0, 0])
    H_dressed_energies.append(E_iqcc)

theta_opt = thetas[np.argmin(H_dressed_energies)]
E_iqcc_opt = min(H_dressed_energies)

print(f'iQCC（1 轮，绕 Y）: {E_iqcc_opt:.6f}  （θ={np.degrees(theta_opt):.1f}°）')
print('单比特下一次 Pauli 旋转即可达到精确基态（教学特例）。')
print()
print('多比特体系（如 OLED 配合物）：')
print('  - 需要多轮 Pauli 旋转（例如数十到上百步）')
print('  - 每步用最优 Pauli 项 dress 哈密顿量')
print('  - 大规模时常在经典端模拟该量子算法流程')
print('  - 2512.13657v2 中 iQCC+PT 对 Ir(III) 等可达化学精度')


---
## 5. InQuanto 平台概览

InQuanto（Quantinuum）面向工业量子化学工作流。
文档：https://docs.quantinuum.com/inquanto/

### 主要能力
- iQCC、iQCC+PT（与 2512.13657v2 等一致）
- UCCSD、ADAPT-VQE
- 多种费米子–量子比特映射
- 与 PySCF、ORCA、NWChem 等对接
- 可对接 Quantinuum H 系列离子阱硬件

### 快速上手（`pip install inquanto` 之后）


In [ ]:
INQUANTO_QUICKSTART = '''
# InQuanto 快速示例：H₂ VQE
# 文档：https://docs.quantinuum.com/inquanto/

import inquanto
from inquanto.computables import ExpectationValue
from inquanto.chemistry import ChemistryDriverPySCF
from inquanto.mappings import JordanWignerMappingFermion
from inquanto.ansatze import FermionSpaceAnsatzUCCSD
from inquanto.algorithms import VQE
from inquanto.minimizers import Scipy

# 分子驱动
driver = ChemistryDriverPySCF.from_molecule(geometry="H 0 0 0; H 0 0 0.74", basis="sto-3g")
hamiltonian, fermion_space, state = driver.get_system()

# 映射到量子比特
qubit_hamiltonian = JordanWignerMappingFermion.operator_map(hamiltonian)

# UCCSD ansatz
ansatz = FermionSpaceAnsatzUCCSD(fermion_space, state)

# 运行 VQE
vqe = VQE(
    ansatz=ansatz,
    observable=qubit_hamiltonian,
    minimizer=Scipy(method="cobyla"),
)
result = vqe.run()
print(f"VQE 能量: {result.value:.6f} Ha")
'''

print('InQuanto 快速上手代码：')
print(INQUANTO_QUICKSTART)

print('\n=== 算法对比（概要）===')
algs = [
    ('VQE+UCCSD',   'NISQ',   '可达化学精度',         'O(N^4) 参数、贫瘠高原风险'),
    ('ADAPT-VQE',   'NISQ',   '可达化学精度',         '参数更少、梯度评估多'),
    ('SQD',         'NISQ',   '逼近 FCI',            '适配噪声硬件，依赖采样质量'),
    ('SKQD',        'NISQ',   '可证明收敛',          '含 qDRIFT 等开销，见 2508.02578v2'),
    ('iQCC',        'NISQ',   '工业可扩展',          '大量在经典端模拟量子比特算符'),
    ('QPE',         '容错 QC', '任意精度（理论上）',   '需容错量子计算（远期）'),
]
print(f'{"算法":<14}{"硬件":<10}{"精度目标":<22}{"主要限制/代价"}')
print('-'*70)
for alg in algs:
    print(f'{alg[0]:<14}{alg[1]:<10}{alg[2]:<22}{alg[3]}')


---
## 6. 练习：精读并批注 2512.13657v2.pdf（iQCC）

学完本笔记本后，建议带着下列问题读 iQCC 论文：

1. **第 2 节（结果）**：iQCC+PT 与 CCSD(T) 在 Ir(III) **T1 能量**上如何对比？
   - 相对实验的 RMSD 多大？
   - 哪些分子偏差最大，可能原因是什么？

2. **第 3 节（方法）**：OLED 配合物的 **活性空间** 如何选取？
   - 用到多少 **逻辑量子比特**？
   - 文中的「量子求解器」与 **真实量子硬件** 有何区别？

3. **与你熟悉的 DFT 对照**：
   - 这类 Ir(III)/Pt(II) 体系正是 TDDFT 常见研究对象
   - DFT 在 **SOC** 与 **T1** 上通常弱在哪里？
   - 若要用 Qiskit + IBM 硬件复现类似思路，还需要哪些环节？

**再读 2508.02578v2.pdf（SKQD）**，可关注：
- qDRIFT 与常规 Trotter 分解有何不同？
- 第 III 节的 **收敛定理** 说了什么条件？
- 文中 **多环芳烃（PAH）** 例子与催化/相关材料问题有何联系？
